# XCCY Swap Pricing - Análisis Detallado

Este notebook analiza paso a paso cómo funciona `XccySwapPricer`:

1. **Inputs** que recibe la función `price()`
2. **Construcción del schedule** de pagos
3. **Amortización** (bullet vs linear vs custom)
4. **Valoración de cada pata** (USD SOFR / COP IBR)
5. **Notional exchange** PV
6. **NPV final** y conversión FX
7. **Mid-life pricing** (swaps ya iniciados)
8. **P&L decomposition** y sensibilidades

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'), override=True)

import QuantLib as ql
import pandas as pd
from datetime import datetime

from pricing.data.market_data import MarketDataLoader
from pricing.curves.curve_manager import CurveManager
from pricing.instruments.xccy_swap import (
    XccySwapPricer,
    build_amortization_schedule,
    validate_amortization_schedule,
)
from utilities.date_functions import datetime_to_ql, ql_to_datetime

pd.set_option('display.float_format', '{:.6f}'.format)

# Construir curvas
loader = MarketDataLoader()
cm = CurveManager()
cm.build_all(loader)
print(f'Valuation date: {cm.valuation_date}')
print(f'FX Spot: {cm.fx_spot}')
print(f'IBR curve: {"OK" if cm.ibr_curve else "MISSING"}')
print(f'SOFR curve: {"OK" if cm.sofr_curve else "MISSING"}')

---
## 1. Inputs de `XccySwapPricer.price()`

La función recibe estos parámetros:

| Parámetro | Tipo | Descripción |
|---|---|---|
| `notional_usd` | float | Nocional en USD |
| `start_date` | date/ql.Date | Fecha inicio del swap |
| `maturity_date` | date/ql.Date | Fecha vencimiento |
| `xccy_basis_bps` | float | Basis spread en bps (pata COP) |
| `pay_usd` | bool | True = pago USD, recibo COP |
| `fx_initial` | float | FX a inception (para notional exchange) |
| `cop_spread_bps` | float | Spread adicional pata COP |
| `usd_spread_bps` | float | Spread pata USD (ej: SOFR - 22bps) |
| `payment_frequency` | ql.Period | Frecuencia de pago (default 3M) |
| `amortization_type` | str | 'bullet', 'linear', 'custom' |
| `amortization_schedule` | list | Factores custom por período |

In [ ]:
# Ejemplo: CCS USD/COP bullet a 5 años
xccy = XccySwapPricer(cm)

# Parámetros del swap
params = {
    'notional_usd': 10_000_000,
    'start_date': datetime(2025, 3, 5),
    'maturity_date': datetime(2030, 3, 5),
    'xccy_basis_bps': 0.0,
    'pay_usd': True,
    'fx_initial': 4150.0,
    'usd_spread_bps': -22.0,   # SOFR - 22bps (típico Bancolombia)
    'cop_spread_bps': 0.0,
    'amortization_type': 'bullet',
}

print('=== Parámetros del Swap ===')
for k, v in params.items():
    print(f'  {k}: {v}')

---
## 2. Construcción del Schedule de Pagos

In [ ]:
# Replicar la construcción interna del schedule
start_ql = datetime_to_ql(params['start_date'])
mat_ql = datetime_to_ql(params['maturity_date'])

cop_cal = cm.ibr_index.fixingCalendar()
usd_cal = cm.sofr_index.fixingCalendar()
joint_cal = ql.JointCalendar(cop_cal, usd_cal)

schedule = ql.Schedule(
    start_ql, mat_ql,
    ql.Period(3, ql.Months),
    joint_cal,
    ql.Following, ql.Following,
    ql.DateGeneration.Forward, False,
)

dates = list(schedule)
print(f'Número de fechas en schedule: {len(dates)}')
print(f'Número de períodos: {len(dates) - 1}')
print()

schedule_df = pd.DataFrame([
    {
        'period': i,
        'start': ql_to_datetime(dates[i-1]).strftime('%Y-%m-%d'),
        'end': ql_to_datetime(dates[i]).strftime('%Y-%m-%d'),
        'days': ql.Actual360().dayCount(dates[i-1], dates[i]),
        'year_frac': ql.Actual360().yearFraction(dates[i-1], dates[i]),
    }
    for i in range(1, len(dates))
])
schedule_df

---
## 3. Amortización: Bullet vs Linear vs Custom

In [ ]:
notional_usd = params['notional_usd']
n_periods = len(dates) - 1

# Bullet
bullet = build_amortization_schedule(schedule, notional_usd, 'bullet')

# Linear
linear = build_amortization_schedule(schedule, notional_usd, 'linear')

# Custom ejemplo: mantiene 100% 8 períodos, luego baja 20% cada trimestre
custom_factors = [1.0]*8 + [0.8, 0.6, 0.4, 0.2] + [0.2]*(n_periods - 12)
custom_factors = custom_factors[:n_periods]
custom = build_amortization_schedule(schedule, notional_usd, 'custom', custom_factors)

amort_df = pd.DataFrame({
    'period': range(1, n_periods+1),
    'bullet': bullet,
    'linear': linear,
    'custom': custom,
})
amort_df['bullet_M'] = amort_df['bullet'] / 1e6
amort_df['linear_M'] = amort_df['linear'] / 1e6
amort_df['custom_M'] = amort_df['custom'] / 1e6
print(f'Períodos: {n_periods}')
amort_df[['period', 'bullet_M', 'linear_M', 'custom_M']]

In [ ]:
# Validar un schedule custom
validation = validate_amortization_schedule(custom_factors)
print('Validación del schedule custom:')
for k, v in validation.items():
    print(f'  {k}: {v}')

---
## 4. Valoración de cada pata (paso a paso)

Para cada período:
1. Forward rate desde la curva
2. + spread
3. Cashflow = notional * (fwd + spread) * tau
4. PV = cashflow * DF(end)

In [ ]:
# Pata USD (SOFR + spread)
usd_spread = params['usd_spread_bps'] / 10_000
cop_spread = (params['xccy_basis_bps'] + params['cop_spread_bps']) / 10_000
fx = params['fx_initial']
dc = ql.Actual360()

usd_notionals = build_amortization_schedule(schedule, notional_usd, 'bullet')
cop_notionals = [n * fx for n in usd_notionals]

usd_leg_rows = []
cop_leg_rows = []

for i in range(1, len(dates)):
    start = dates[i-1]
    end = dates[i]
    tau = dc.yearFraction(start, end)
    
    # USD
    fwd_usd = cm.sofr_handle.forwardRate(start, end, dc, ql.Simple).rate()
    cf_usd = usd_notionals[i-1] * (fwd_usd + usd_spread) * tau
    df_usd = cm.sofr_handle.discount(end)
    pv_usd = cf_usd * df_usd
    
    usd_leg_rows.append({
        'period': i,
        'start': ql_to_datetime(start).strftime('%Y-%m-%d'),
        'end': ql_to_datetime(end).strftime('%Y-%m-%d'),
        'notional': usd_notionals[i-1],
        'fwd_rate': fwd_usd * 100,
        'spread': usd_spread * 100,
        'all_in_rate': (fwd_usd + usd_spread) * 100,
        'tau': tau,
        'cashflow': cf_usd,
        'df': df_usd,
        'pv': pv_usd,
    })
    
    # COP
    fwd_cop = cm.ibr_handle.forwardRate(start, end, dc, ql.Simple).rate()
    cf_cop = cop_notionals[i-1] * (fwd_cop + cop_spread) * tau
    df_cop = cm.ibr_handle.discount(end)
    pv_cop = cf_cop * df_cop
    
    cop_leg_rows.append({
        'period': i,
        'start': ql_to_datetime(start).strftime('%Y-%m-%d'),
        'end': ql_to_datetime(end).strftime('%Y-%m-%d'),
        'notional': cop_notionals[i-1],
        'fwd_rate': fwd_cop * 100,
        'spread': cop_spread * 100,
        'all_in_rate': (fwd_cop + cop_spread) * 100,
        'tau': tau,
        'cashflow': cf_cop,
        'df': df_cop,
        'pv': pv_cop,
    })

usd_leg_df = pd.DataFrame(usd_leg_rows)
cop_leg_df = pd.DataFrame(cop_leg_rows)

print('=== PATA USD (SOFR + spread) ===')
print(f'USD spread: {usd_spread*10000:.0f} bps')
print(f'PV total pata USD: {usd_leg_df["pv"].sum():,.2f} USD')
usd_leg_df

In [ ]:
print('=== PATA COP (IBR + basis + spread) ===')
print(f'COP spread total: {cop_spread*10000:.0f} bps')
print(f'PV total pata COP: {cop_leg_df["pv"].sum():,.2f} COP')
cop_leg_df

---
## 5. Notional Exchange PV

Para un swap bullet:
- **Inicio**: Pago USD notional / Recibo COP notional
- **Final**: Recibo USD notional / Pago COP notional

Para un swap amortizable se agregan retornos intermedios.

In [ ]:
# Notional exchange PV (bullet)
notional_cop = notional_usd * fx

# Desde la perspectiva del payer USD:
# Inicio: paga USD (negativo), recibe COP (positivo) — ya liquidado si mid-life
# Final: recibe USD (positivo), paga COP (negativo)

df_start_usd = cm.sofr_handle.discount(dates[0])
df_end_usd = cm.sofr_handle.discount(dates[-1])
df_start_cop = cm.ibr_handle.discount(dates[0])
df_end_cop = cm.ibr_handle.discount(dates[-1])

print('=== Discount Factors para Notional Exchange ===')
print(f'Start ({ql_to_datetime(dates[0]).strftime("%Y-%m-%d")}):')
print(f'  SOFR DF: {df_start_usd:.8f}')
print(f'  IBR  DF: {df_start_cop:.8f}')
print(f'End ({ql_to_datetime(dates[-1]).strftime("%Y-%m-%d")}):')
print(f'  SOFR DF: {df_end_usd:.8f}')
print(f'  IBR  DF: {df_end_cop:.8f}')
print()

# La función internamente usa _notional_exchange_pv_amort
# Para bullet: PV = -N*DF(start) + N*DF(end)
usd_nex_pv_raw = -notional_usd * df_start_usd + notional_usd * df_end_usd
cop_nex_pv_raw = -notional_cop * df_start_cop + notional_cop * df_end_cop

# El pricer niega estos (convención interna)
usd_nex_pv = -usd_nex_pv_raw
cop_nex_pv = -cop_nex_pv_raw

print('=== Notional Exchange PV ===')
print(f'USD notional exchange PV: {usd_nex_pv:,.2f} USD')
print(f'COP notional exchange PV: {cop_nex_pv:,.2f} COP')

---
## 6. NPV Final y Conversión FX

```
usd_total = usd_leg_pv + usd_notional_exchange_pv
cop_total = cop_leg_pv + cop_notional_exchange_pv

sign = +1 si pay_usd, -1 si receive_usd
NPV_COP = sign * (-usd_total * fx_spot + cop_total)
NPV_USD = NPV_COP / fx_spot
```

In [ ]:
usd_leg_pv = usd_leg_df['pv'].sum()
cop_leg_pv = cop_leg_df['pv'].sum()

usd_total = usd_leg_pv + usd_nex_pv
cop_total = cop_leg_pv + cop_nex_pv

spot = cm.fx_spot
sign = 1.0  # pay_usd = True

npv_cop = sign * (-usd_total * spot + cop_total)
npv_usd = npv_cop / spot

print('=== Componentes del NPV ===')
print(f'USD leg PV:              {usd_leg_pv:>20,.2f} USD')
print(f'USD notional exch PV:    {usd_nex_pv:>20,.2f} USD')
print(f'USD total:               {usd_total:>20,.2f} USD')
print()
print(f'COP leg PV:              {cop_leg_pv:>20,.2f} COP')
print(f'COP notional exch PV:    {cop_nex_pv:>20,.2f} COP')
print(f'COP total:               {cop_total:>20,.2f} COP')
print()
print(f'FX spot:                 {spot:>20,.2f}')
print(f'-USD_total * spot:       {-usd_total * spot:>20,.2f} COP')
print()
print(f'NPV (COP):               {npv_cop:>20,.2f}')
print(f'NPV (USD):               {npv_usd:>20,.2f}')

---
## 7. Comparar con `xccy.price()` (debe dar igual)

In [ ]:
result = xccy.price(**params)
print('=== Output de xccy.price() ===')
for k, v in result.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:>20,.4f}')
    else:
        print(f'  {k:30s}: {v}')

print()
print('=== Verificación ===')
print(f'NPV manual vs pricer (COP): {npv_cop:,.2f} vs {result["npv_cop"]:,.2f}')
print(f'Diferencia: {abs(npv_cop - result["npv_cop"]):,.2f}')

---
## 8. Par Xccy Basis

Encontrar el spread (bps) que hace NPV = 0.

In [ ]:
par_basis = xccy.par_xccy_basis(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    fx_initial=4150.0,
)
print(f'Par XCCY basis: {par_basis:.2f} bps')

# Verificar: repricear con par basis debería dar NPV ~ 0
check = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=par_basis,
    fx_initial=4150.0,
    pay_usd=True,
)
print(f'NPV con par basis: {check["npv_cop"]:,.2f} COP (debe ser ~0)')

---
## 9. Mid-Life Pricing

Para swaps donde `start_date < evaluation_date` (ya están corriendo),
el pricer:
- Construye el schedule completo desde `start_date`
- Filtra períodos pasados
- Clip la primera fecha futura al `curve_floor`
- No incluye el notional exchange inicial (ya se liquidó)

In [ ]:
# Swap que empezó hace 1 año
midlife_result = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2024, 3, 5),   # empezó hace ~1 año
    maturity_date=datetime(2029, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=3900.0,                  # FX al momento de inception
    usd_spread_bps=-22.0,
)

print('=== Mid-Life Swap (start 2024-03-05, mat 2029-03-05) ===')
print(f'FX inception: {midlife_result["fx_initial"]:,.2f}')
print(f'FX spot:      {midlife_result["fx_spot"]:,.2f}')
print(f'NPV (COP):    {midlife_result["npv_cop"]:>20,.2f}')
print(f'NPV (USD):    {midlife_result["npv_usd"]:>20,.2f}')
print(f'USD leg PV:   {midlife_result["usd_leg_pv"]:>20,.4f}')
print(f'COP leg PV:   {midlife_result["cop_leg_pv"]:>20,.4f}')
print(f'USD nex PV:   {midlife_result["usd_notional_exchange_pv"]:>20,.4f}')
print(f'COP nex PV:   {midlife_result["cop_notional_exchange_pv"]:>20,.4f}')

---
## 10. P&L Decomposition (FX vs Rates)

In [ ]:
pnl = xccy.pnl_inception(
    notional_usd=10_000_000,
    start_date=datetime(2024, 3, 5),
    maturity_date=datetime(2029, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=3900.0,
    usd_spread_bps=-22.0,
)

print('=== P&L Decomposition ===')
print(f'NPV total (COP):     {pnl["npv_cop"]:>20,.2f}')
print(f'P&L FX (COP):        {pnl["pnl_fx_cop"]:>20,.2f}')
print(f'P&L Rates (COP):     {pnl["pnl_rates_cop"]:>20,.2f}')
print()
print(f'FX inception:         {pnl["fx_initial"]:>20,.2f}')
print(f'FX spot:              {pnl["fx_spot"]:>20,.2f}')
print(f'FX move:              {pnl["fx_spot"] - pnl["fx_initial"]:>+20,.2f}')

---
## 11. Sensibilidades (DV01 IBR y SOFR)

In [ ]:
# Base NPV
base = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)
base_npv = base['npv_cop']

# Bump IBR +1bp
cm.bump_ibr(1)
ibr_bumped = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)
dv01_ibr = ibr_bumped['npv_cop'] - base_npv
cm.bump_ibr(-1)  # reset

# Bump SOFR +1bp
cm.bump_sofr(1)
sofr_bumped = xccy.price(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)
dv01_sofr = sofr_bumped['npv_cop'] - base_npv
cm.bump_sofr(-1)  # reset

print('=== DV01 (sensibilidad a +1bp) ===')
print(f'Base NPV (COP):   {base_npv:>20,.2f}')
print(f'DV01 IBR (COP):   {dv01_ibr:>20,.2f}')
print(f'DV01 SOFR (COP):  {dv01_sofr:>20,.2f}')
print(f'DV01 Total (COP): {dv01_ibr + dv01_sofr:>20,.2f}')

---
## 12. Comparar Bullet vs Amortizable

In [ ]:
common_params = dict(
    notional_usd=10_000_000,
    start_date=datetime(2025, 3, 5),
    maturity_date=datetime(2030, 3, 5),
    xccy_basis_bps=0.0,
    pay_usd=True,
    fx_initial=4150.0,
    usd_spread_bps=-22.0,
)

bullet_result = xccy.price(**common_params, amortization_type='bullet')
linear_result = xccy.price(**common_params, amortization_type='linear')

comparison = pd.DataFrame([
    {
        'type': 'Bullet',
        'npv_cop': bullet_result['npv_cop'],
        'npv_usd': bullet_result['npv_usd'],
        'usd_leg_pv': bullet_result['usd_leg_pv'],
        'cop_leg_pv': bullet_result['cop_leg_pv'],
        'usd_nex_pv': bullet_result['usd_notional_exchange_pv'],
        'cop_nex_pv': bullet_result['cop_notional_exchange_pv'],
    },
    {
        'type': 'Linear',
        'npv_cop': linear_result['npv_cop'],
        'npv_usd': linear_result['npv_usd'],
        'usd_leg_pv': linear_result['usd_leg_pv'],
        'cop_leg_pv': linear_result['cop_leg_pv'],
        'usd_nex_pv': linear_result['usd_notional_exchange_pv'],
        'cop_nex_pv': linear_result['cop_notional_exchange_pv'],
    },
]).set_index('type')

print('=== Bullet vs Linear Amortization ===')
comparison

In [ ]:
# Par basis para cada tipo
par_bullet = xccy.par_xccy_basis(**common_params, amortization_type='bullet')
par_linear = xccy.par_xccy_basis(**common_params, amortization_type='linear')

print(f'Par basis (Bullet): {par_bullet:+.2f} bps')
print(f'Par basis (Linear): {par_linear:+.2f} bps')
print(f'Diferencia:         {par_linear - par_bullet:+.2f} bps')